In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/pipeline/roadsign_data_kaggle.yaml
/kaggle/input/pipeline/resnet50_best.pth
/kaggle/input/pipeline/best.pt


In [2]:
!find /kaggle/input -type f | grep -E "best\.pt|resnet50_best\.pth|ya?ml" | head -n 50


/kaggle/input/pipeline/roadsign_data_kaggle.yaml
/kaggle/input/pipeline/resnet50_best.pth
/kaggle/input/pipeline/best.pt


In [3]:
%cd /kaggle/working
!git clone https://github.com/WongKinYiu/yolov7.git
%cd /kaggle/working/yolov7
!pip -q install -r requirements.txt


/kaggle/working
Cloning into 'yolov7'...
remote: Enumerating objects: 1197, done.
remote: Total 1197 (delta 0), reused 0 (delta 0), pack-reused 1197 (from 1)
Receiving objects: 100% (1197/1197), 74.29 MiB | 23.06 MiB/s, done.
Resolving deltas: 100% (511/511), done.
/kaggle/working/yolov7
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 98.4 MB/s eta 0:00:0000:010:01
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.


In [4]:
%cd /kaggle/working/yolov7
!python -m pip -q install --upgrade pip setuptools wheel
!pip -q install thop


/kaggle/working/yolov7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
bigframes 2.26.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
fastai 2.8.4 requires fastcore<1.9,>=1.8.0, but you have fastcore 1.11.3 which is incompatible.


In [5]:
import sys, os
sys.path.insert(0, "/kaggle/working/yolov7")

from models.experimental import attempt_load
from utils.general import non_max_suppression, scale_coords
from utils.datasets import letterbox

print("✅ YOLOv7 imports successful")


✅ YOLOv7 imports successful


In [6]:
import torch
ckpt = torch.load("/kaggle/input/pipeline/resnet50_best.pth", map_location="cpu")
print(type(ckpt))
if isinstance(ckpt, dict):
    print("keys:", ckpt.keys())


<class 'dict'>
keys: dict_keys(['model_state', 'classes'])


In [7]:
import yaml, torch

# YOLO names
with open("/kaggle/input/pipeline/roadsign_data_kaggle.yaml", "r") as f:
    y = yaml.safe_load(f)
yolo_names = y["names"]
if isinstance(yolo_names, dict):
    yolo_names = [yolo_names[k] for k in sorted(yolo_names.keys())]

# ResNet names
ckpt = torch.load("/kaggle/input/pipeline/resnet50_best.pth", map_location="cpu")
resnet_names = ckpt["classes"]

print("YOLO classes:", len(yolo_names))
print("ResNet classes:", len(resnet_names))
print("Same order:", yolo_names == resnet_names)

# show a few examples
print("\nFirst 10 YOLO:", yolo_names[:10])
print("First 10 ResNet:", resnet_names[:10])


YOLO classes: 29
ResNet classes: 29
Same order: True

First 10 YOLO: ['Crossroads', 'Emergency Stopping', 'Emergency Stopping 250m', 'Give Way', 'Height Limit 5-7m', 'Hospital Ahead', 'Junction Ahead', 'Mosque Ahead', 'No Overtaking', 'No Pedestrians']
First 10 ResNet: ['Crossroads', 'Emergency Stopping', 'Emergency Stopping 250m', 'Give Way', 'Height Limit 5-7m', 'Hospital Ahead', 'Junction Ahead', 'Mosque Ahead', 'No Overtaking', 'No Pedestrians']


In [8]:
!find /kaggle/input -type f | grep -Ei "\.(jpg|jpeg|png)$" | head -n 30

/kaggle/input/v7-pytorch-for-hybrid/valid/images/Left_lane_for_truck_64_jpg.rf.c17232a2da0f29e9d2afc4c904e44f69.jpg
/kaggle/input/v7-pytorch-for-hybrid/valid/images/Tolls_ahead_25_jpg.rf.396a7980a0fad5965ad83790614950ea.jpg
/kaggle/input/v7-pytorch-for-hybrid/valid/images/Left_turn_75_jpg.rf.832ae50b89056dd14a951bb352e2eb87.jpg
/kaggle/input/v7-pytorch-for-hybrid/valid/images/Height_limit_5-7m_465_jpg.rf.504765c146d4c81a1c43d5c0195a785d.jpg
/kaggle/input/v7-pytorch-for-hybrid/valid/images/Side_road_on_left_174_jpg.rf.de3cb07fae4b089f2ee6f48a8fb8ff80.jpg
/kaggle/input/v7-pytorch-for-hybrid/valid/images/Side_road_on_left_730_jpg.rf.43f685a826403a19674f610bf0f7530c.jpg
/kaggle/input/v7-pytorch-for-hybrid/valid/images/Speed_breaker_106_jpg.rf.1db90bd056fda5b35219489aacab3789.jpg
/kaggle/input/v7-pytorch-for-hybrid/valid/images/Speed_limit_80km_298_jpg.rf.9d0cfb7df476354c5c6e85cd5c4c3ab4.jpg
/kaggle/input/v7-pytorch-for-hybrid/valid/images/Side_road_on_right_242_jpg.rf.2b1ed777390dcd1edde1c

In [12]:
TEST_IMG = "/kaggle/input/v7-pytorch-for-hybrid/valid/images/School_ahead_226_jpg.rf.cdb835115dc047d900799c7819f5208e.jpg"


In [14]:
import os, re

exp_path = "/kaggle/working/yolov7/models/experimental.py"
text = open(exp_path, "r", encoding="utf-8").read()

# Replace ONLY the torch.load line inside attempt_load
text2 = text.replace(
    "ckpt = torch.load(w, map_location=map_location)  # load",
    "ckpt = torch.load(w, map_location=map_location, weights_only=False)  # load"
)

if text2 == text:
    print("⚠️ Patch did not apply (line not found). Open experimental.py and search for torch.load.")
else:
    open(exp_path, "w", encoding="utf-8").write(text2)
    print("✅ Patched YOLOv7 attempt_load to use weights_only=False")


✅ Patched YOLOv7 attempt_load to use weights_only=False


In [15]:
!grep -n "torch.load" /kaggle/working/yolov7/models/experimental.py | head -n 20


252:        ckpt = torch.load(w, map_location=map_location, weights_only=False)  # load


In [18]:
import numpy as np
import torch

# allow numpy reconstruct for trusted checkpoints
torch.serialization.add_safe_globals([np.core.multiarray._reconstruct])

print("✅ Added safe global for numpy reconstruct")


✅ Added safe global for numpy reconstruct


In [21]:
import numpy as np
import torch

torch.serialization.add_safe_globals([
    np.core.multiarray._reconstruct,
    np.ndarray,
    np.dtype
])

print("✅ Added safe globals: reconstruct, ndarray, dtype")


✅ Added safe globals: reconstruct, ndarray, dtype


In [28]:
from pathlib import Path

exp_path = Path("/kaggle/working/yolov7/models/experimental.py")
txt = exp_path.read_text(encoding="utf-8")

old = "        ckpt = torch.load(w, map_location=map_location, weights_only=False)  # load"

new = (
"        import numpy as np\n"
"        _SAFE = [\n"
"            np.core.multiarray._reconstruct,\n"
"            np.core.multiarray.scalar,\n"
"            np.ndarray,\n"
"            np.dtype,\n"
"        ]\n"
"        with torch.serialization.safe_globals(_SAFE):\n"
"            ckpt = torch.load(w, map_location=map_location, weights_only=False)  # load"
)

if old not in txt:
    print("⚠️ Patch target line not found. Run this to inspect:\n!grep -n \"ckpt = torch.load\" /kaggle/working/yolov7/models/experimental.py")
else:
    exp_path.write_text(txt.replace(old, new), encoding="utf-8")
    print("✅ Patched experimental.py to load best.pt with safe_globals")


✅ Patched experimental.py to load best.pt with safe_globals


In [30]:
import torch
import numpy as np
import numpy._core.multiarray as nma

BEST = "/kaggle/input/pipeline/best.pt"

SAFE = [
    nma._reconstruct,   # <-- the exact one from the error
    nma.scalar,
    np.ndarray,
    np.dtype,
]

try:
    with torch.serialization.safe_globals(SAFE):
        ckpt = torch.load(BEST, map_location="cpu", weights_only=False)
    print("✅ Manual torch.load(best.pt) OK")
    print("ckpt keys:", ckpt.keys() if isinstance(ckpt, dict) else type(ckpt))
except Exception as e:
    print("❌ Manual load still failed:\n", e)


✅ Manual torch.load(best.pt) OK
ckpt keys: dict_keys(['epoch', 'best_fitness', 'training_results', 'model', 'ema', 'updates', 'optimizer', 'wandb_id'])


In [32]:
from pathlib import Path

exp_path = Path("/kaggle/working/yolov7/models/experimental.py")
txt = exp_path.read_text(encoding="utf-8")

target = "ckpt = torch.load(w, map_location=map_location, weights_only=False)  # load"

replacement = (
"import numpy as np\n"
"import numpy._core.multiarray as nma\n"
"_SAFE = [nma._reconstruct, nma.scalar, np.ndarray, np.dtype]\n"
"with torch.serialization.safe_globals(_SAFE):\n"
"    ckpt = torch.load(w, map_location=map_location, weights_only=False)  # load"
)

if target not in txt:
    print("⚠️ Could not find the exact torch.load line. Run:\n!grep -n \"ckpt = torch.load\" /kaggle/working/yolov7/models/experimental.py")
else:
    txt2 = txt.replace(target, replacement)
    exp_path.write_text(txt2, encoding="utf-8")
    print("✅ Patched attempt_load with numpy._core safe_globals")


✅ Patched attempt_load with numpy._core safe_globals


In [41]:
from pathlib import Path
import re

exp_path = Path("/kaggle/working/yolov7/models/experimental.py")
txt = exp_path.read_text(encoding="utf-8")

# Find the torch.load line inside attempt_load and replace the whole block around it safely.
pattern = r"ckpt\s*=\s*torch\.load\(w,\s*map_location=map_location.*?\)\s*# load"

replacement = (
"import numpy._core.multiarray as nma\n"
"_SAFE = [nma._reconstruct]\n"
"with torch.serialization.safe_globals(_SAFE):\n"
"    ckpt = torch.load(w, map_location=map_location, weights_only=False)  # load"
)

txt2, n = re.subn(pattern, replacement, txt, count=1)

if n == 0:
    print("⚠️ Could not patch. Show the load line with:\n!grep -n \"ckpt = torch.load\" /kaggle/working/yolov7/models/experimental.py")
else:
    exp_path.write_text(txt2, encoding="utf-8")
    print("✅ Patched experimental.py with numpy._core.multiarray._reconstruct")


✅ Patched experimental.py with numpy._core.multiarray._reconstruct


In [43]:
import torch
try:
    torch.load("/kaggle/input/pipeline/best.pt", map_location="cpu", weights_only=False)
except Exception as e:
    print(e)


In [45]:
import torch, numpy as np

# Save original torch.load
_torch_load = torch.load

def torch_load_unsafe(*args, **kwargs):
    # Force full load
    kwargs["weights_only"] = False
    # Allow numpy reconstruct during unpickling
    with torch.serialization.safe_globals([np.core.multiarray._reconstruct, np.ndarray, np.dtype]):
        return _torch_load(*args, **kwargs)

# Temporarily override
torch.load = torch_load_unsafe

# Now import YOLO and load model
import sys
sys.path.insert(0, "/kaggle/working/yolov7")
from models.experimental import attempt_load

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
yolo = attempt_load("/kaggle/input/pipeline/best.pt", map_location=device)
print("✅ YOLO loaded OK")

# Restore torch.load (important)
torch.load = _torch_load


Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
✅ YOLO loaded OK


/usr/local/lib/python3.12/dist-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4322.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


In [46]:
import torch, numpy as np, sys

# patch torch.load temporarily
_torch_load = torch.load
def torch_load_unsafe(*args, **kwargs):
    kwargs["weights_only"] = False
    with torch.serialization.safe_globals([np.core.multiarray._reconstruct, np.ndarray, np.dtype]):
        return _torch_load(*args, **kwargs)

torch.load = torch_load_unsafe

sys.path.insert(0, "/kaggle/working/yolov7")
from models.experimental import attempt_load

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
yolo = attempt_load("/kaggle/input/pipeline/best.pt", map_location=device).eval()

# restore torch.load
torch.load = _torch_load

print("✅ YOLO ready")


Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
✅ YOLO ready


In [47]:
import yaml
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
import numpy as np

class HybridDetector2:
    def __init__(self, yolo_model, resnet_ckpt, data_yaml, device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.device = torch.device(self.device)

        # YOLO model (already loaded)
        self.yolo = yolo_model.to(self.device).eval()

        # YOLO utils
        from utils.general import non_max_suppression, scale_coords
        from utils.datasets import letterbox
        self.non_max_suppression = non_max_suppression
        self.scale_coords = scale_coords
        self.letterbox = letterbox

        # names
        with open(data_yaml, "r") as f:
            y = yaml.safe_load(f)
        self.names = y["names"]
        if isinstance(self.names, dict):
            self.names = [self.names[k] for k in sorted(self.names.keys())]

        # ResNet
        ckpt = torch.load(resnet_ckpt, map_location="cpu")
        self.cls_names = ckpt["classes"]
        num_classes = len(self.cls_names)

        self.resnet = models.resnet50(weights=None)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)
        self.resnet.load_state_dict(ckpt["model_state"], strict=True)
        self.resnet.to(self.device).eval()

        self.tf = transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
        ])

    @torch.no_grad()
    def detect(self, img_path, img_size=640, conf=0.25, iou=0.45):
        img0 = np.array(Image.open(img_path).convert("RGB"))
        h0, w0 = img0.shape[:2]

        img = self.letterbox(img0, img_size, stride=32, auto=False)[0]
        img = img.transpose((2,0,1))
        img = np.ascontiguousarray(img)

        img = torch.from_numpy(img).to(self.device).float() / 255.0
        if img.ndimension() == 3:
            img = img.unsqueeze(0)

        pred = self.yolo(img, augment=False)[0]
        pred = self.non_max_suppression(pred, conf_thres=conf, iou_thres=iou)[0]

        results = []
        if pred is None or len(pred) == 0:
            return results

        pred[:, :4] = self.scale_coords(img.shape[2:], pred[:, :4], (h0, w0))

        pil0 = Image.fromarray(img0)
        for det in pred:
            x1, y1, x2, y2, conf_det, cls_det = det.tolist()
            x1,y1,x2,y2 = map(int, [x1,y1,x2,y2])

            x1 = max(0, min(x1, w0-1))
            y1 = max(0, min(y1, h0-1))
            x2 = max(0, min(x2, w0-1))
            y2 = max(0, min(y2, h0-1))
            if x2 <= x1 or y2 <= y1:
                continue

            crop = pil0.crop((x1,y1,x2,y2))
            x = self.tf(crop).unsqueeze(0).to(self.device)
            logits = self.resnet(x)
            prob = torch.softmax(logits, dim=1)[0]
            cls2 = int(prob.argmax().item())
            conf2 = float(prob[cls2].item())

            cls_det = int(cls_det)
            results.append({
                "bbox": [x1,y1,x2,y2],
                "yolo_cls": cls_det,
                "yolo_name": self.names[cls_det],
                "yolo_conf": float(conf_det),
                "resnet_cls": cls2,
                "resnet_name": self.cls_names[cls2],
                "resnet_conf": conf2
            })

        return results

detector = HybridDetector2(
    yolo_model=yolo,
    resnet_ckpt="/kaggle/input/pipeline/resnet50_best.pth",
    data_yaml="/kaggle/input/pipeline/roadsign_data_kaggle.yaml",
)
print("✅ HybridDetector ready")


✅ HybridDetector ready


In [48]:
!ls /kaggle/input/v7-pytorch-for-hybrid/test/images | head -n 20


Emergency_break_0_jpg.rf.a2e77f5e7482396445b6d7971adec7fc.jpg
Emergency_break_104_jpg.rf.8dde8adefe9f17da98f612068b58b952.jpg
Emergency_break_104_jpg.rf.edccc89ef9bea669cec841414e50b4e3.jpg
Emergency_break_109_jpg.rf.d99e0c64d7be20bf55202fea587e1f91.jpg
Emergency_break_11_jpg.rf.1e484bae44469155694b39af8d1e3e68.jpg
Emergency_break_122_jpg.rf.647ae466b92535ff5b402bd798e9cfd6.jpg
Emergency_break_123_jpg.rf.273fdf40db5a34a8cc6f27ab4f82d435.jpg
Emergency_break_128_jpg.rf.584a079143a1dac7247be21b9d06296d.jpg
Emergency_break_12_jpg.rf.96cf528ec76a64a98e6f6d4280e3ac7a.jpg
Emergency_break_131_jpg.rf.f7a9ac84b26640a916f602f5cbfffc33.jpg
Emergency_break_134_jpg.rf.891683eb24b485bdadff80eb44949837.jpg
Emergency_break_135_jpg.rf.0032de6c848f8282fab2dc36acf1e843.jpg
Emergency_break_138_jpg.rf.4690b34f13fdd68251f9429cf9f0e023.jpg
Emergency_break_22_jpg.rf.5c3f255ae3b194bd8a755391ed70a969.jpg
Emergency_break_23_jpg.rf.3ada43bdde6d42d4788025af16ca70a4.jpg
Emergency_break_250m_0_jpg.rf.39e3d4965daa35ab

In [49]:
results = detector.detect(TEST_IMG, conf=0.25, iou=0.45)
print("detections:", len(results))
results[:5]


detections: 1


[{'bbox': [154, 103, 438, 320],
  'yolo_cls': 13,
  'yolo_name': 'School Ahead',
  'yolo_conf': 0.9639951586723328,
  'resnet_cls': 13,
  'resnet_name': 'School Ahead',
  'resnet_conf': 0.9998052716255188}]

In [50]:
from PIL import Image, ImageDraw

img = Image.open(TEST_IMG).convert("RGB")
draw = ImageDraw.Draw(img)

for r in results:
    x1,y1,x2,y2 = r["bbox"]
    label = f'{r["yolo_name"]}({r["yolo_conf"]:.2f}) | {r["resnet_name"]}({r["resnet_conf"]:.2f})'
    draw.rectangle([x1,y1,x2,y2], width=3)
    draw.text((x1, max(0, y1-12)), label)

out_path = "/kaggle/working/hybrid_pred.jpg"
img.save(out_path)
out_path


'/kaggle/working/hybrid_pred.jpg'

In [51]:
import os, glob, json, pickle

IMG_DIR = "/kaggle/input/v7-pytorch-for-hybrid/test/images"
paths = sorted(glob.glob(os.path.join(IMG_DIR, "*.jpg")) + glob.glob(os.path.join(IMG_DIR, "*.png")))

all_results = {}
for p in paths:
    all_results[os.path.basename(p)] = detector.detect(p, conf=0.25, iou=0.45)

os.makedirs("/kaggle/working/hybrid_outputs", exist_ok=True)

with open("/kaggle/working/hybrid_outputs/results.json", "w") as f:
    json.dump(all_results, f, indent=2)

with open("/kaggle/working/hybrid_outputs/results.pkl", "wb") as f:
    pickle.dump(all_results, f)

print("✅ Saved to /kaggle/working/hybrid_outputs/")
print("Total images:", len(paths))


✅ Saved to /kaggle/working/hybrid_outputs/
Total images: 999


In [54]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

labels_all = list(range(len(classes)))  # 0..28

print(classification_report(
    y_true, y_pred,
    labels=labels_all,
    target_names=classes,
    digits=4,
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred, labels=labels_all)


                           precision    recall  f1-score   support

               Crossroads     1.0000    1.0000    1.0000        21
       Emergency Stopping     1.0000    1.0000    1.0000        34
  Emergency Stopping 250m     1.0000    1.0000    1.0000        24
                 Give Way     1.0000    1.0000    1.0000        19
        Height Limit 5-7m     1.0000    1.0000    1.0000        54
           Hospital Ahead     1.0000    1.0000    1.0000        25
           Junction Ahead     1.0000    1.0000    1.0000         8
             Mosque Ahead     1.0000    1.0000    1.0000        70
            No Overtaking     1.0000    1.0000    1.0000        64
           No Pedestrians     1.0000    1.0000    1.0000        15
         No Vehicle Entry     1.0000    1.0000    1.0000        23
     Pedestrians Crossing     1.0000    1.0000    1.0000        66
        Petrol Pump Ahead     1.0000    1.0000    1.0000        75
             School Ahead     1.0000    1.0000    1.0000     

In [58]:
!du -sh /kaggle/working/hybrid_outputs


516K	/kaggle/working/hybrid_outputs


In [60]:
IMG_DIR = "/kaggle/input/v7-pytorch-for-hybrid/test/images"


In [61]:
import os, glob, random, zipfile
from PIL import Image, ImageDraw

# --- CONFIG ---
IMG_DIR = "/kaggle/input/v7-pytorch-for-hybrid/test/images"
OUT_DIR = "/kaggle/working/hybrid_20_outputs"
ZIP_PATH = "/kaggle/working/hybrid_20_outputs.zip"
N = 20

os.makedirs(OUT_DIR, exist_ok=True)

# collect images
paths = sorted(glob.glob(os.path.join(IMG_DIR, "*.jpg")) + glob.glob(os.path.join(IMG_DIR, "*.png")))
assert len(paths) > 0, "No images found in IMG_DIR"

# pick 20 (random)
random.seed(42)
pick = random.sample(paths, min(N, len(paths)))

saved = []
for p in pick:
    img = Image.open(p).convert("RGB")
    results = detector.detect(p, conf=0.25, iou=0.45)

    draw = ImageDraw.Draw(img)
    for r in results:
        x1,y1,x2,y2 = r["bbox"]
        text = f'{r["yolo_name"]} {r["yolo_conf"]:.2f} | {r["resnet_name"]} {r["resnet_conf"]:.2f}'
        draw.rectangle([x1,y1,x2,y2], width=3)
        draw.text((x1, max(0, y1-12)), text)

    out_name = os.path.basename(p)
    out_path = os.path.join(OUT_DIR, out_name)
    img.save(out_path)
    saved.append(out_path)

print("✅ Saved annotated images:", len(saved))
print("Output folder:", OUT_DIR)

# make zip
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in saved:
        z.write(f, arcname=os.path.basename(f))

print("✅ Zip created:", ZIP_PATH)


✅ Saved annotated images: 20
Output folder: /kaggle/working/hybrid_20_outputs
✅ Zip created: /kaggle/working/hybrid_20_outputs.zip


In [62]:
import os, glob, random, zipfile
from PIL import Image, ImageDraw

# --- CONFIG ---
IMG_DIR = "/kaggle/input/v7-pytorch-for-hybrid/test/images"
OUT_DIR = "/kaggle/working/hybrid_50_outputs"
ZIP_PATH = "/kaggle/working/hybrid_50_outputs.zip"
N = 50

os.makedirs(OUT_DIR, exist_ok=True)

# collect images
paths = sorted(glob.glob(os.path.join(IMG_DIR, "*.jpg")) + glob.glob(os.path.join(IMG_DIR, "*.png")))
assert len(paths) > 0, "No images found in IMG_DIR"

# pick 20 (random)
random.seed(42)
pick = random.sample(paths, min(N, len(paths)))

saved = []
for p in pick:
    img = Image.open(p).convert("RGB")
    results = detector.detect(p, conf=0.25, iou=0.45)

    draw = ImageDraw.Draw(img)
    for r in results:
        x1,y1,x2,y2 = r["bbox"]
        text = f'{r["yolo_name"]} {r["yolo_conf"]:.2f} | {r["resnet_name"]} {r["resnet_conf"]:.2f}'
        draw.rectangle([x1,y1,x2,y2], width=3)
        draw.text((x1, max(0, y1-12)), text)

    out_name = os.path.basename(p)
    out_path = os.path.join(OUT_DIR, out_name)
    img.save(out_path)
    saved.append(out_path)

print("✅ Saved annotated images:", len(saved))
print("Output folder:", OUT_DIR)

# make zip
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in saved:
        z.write(f, arcname=os.path.basename(f))

print("✅ Zip created:", ZIP_PATH)


✅ Saved annotated images: 50
Output folder: /kaggle/working/hybrid_50_outputs
✅ Zip created: /kaggle/working/hybrid_50_outputs.zip


In [ ]:
VIDEO_IN = "/kaggle/input/videodetectbyhybrid/1.mp4"


In [64]:
import os
import cv2

VIDEO_IN  = "/kaggle/input/videodetectbyhybrid/1.mp4"
VIDEO_OUT = "/kaggle/working/hybrid_output.mp4"

conf = 0.25
iou  = 0.45

cap = cv2.VideoCapture(VIDEO_IN)
assert cap.isOpened(), f"Could not open video: {VIDEO_IN}"

fps = cap.get(cv2.CAP_PROP_FPS) or 25
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (W, H))

frame_i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break

    # --- HYBRID DETECT (uses your detector) ---
    results = detector.detect_frame(frame, conf=conf, iou=iou) if hasattr(detector, "detect_frame") else None

    # If your detector only supports image-path detect(), use this fallback:
    # We'll run via a helper function defined below
    if results is None:
        results = detect_bgr_with_detector(detector, frame, conf=conf, iou=iou)

    # draw
    for r in results:
        x1,y1,x2,y2 = r["bbox"]
        label = f'{r["yolo_name"]} {r["yolo_conf"]:.2f} | {r["resnet_name"]} {r["resnet_conf"]:.2f}'
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(frame, label, (x1, max(20, y1-7)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

    writer.write(frame)

    frame_i += 1
    if frame_i % 50 == 0:
        print("processed frames:", frame_i)

cap.release()
writer.release()

print("✅ Saved:", VIDEO_OUT)


NameError: name 'detect_bgr_with_detector' is not defined

In [66]:
import os
import cv2
import numpy as np
import torch
from PIL import Image

# -------------------
# SET YOUR VIDEO PATH
# -------------------
VIDEO_IN  = "/kaggle/input/videodetectbyhybrid/1.mp4"  # <-- change this
VIDEO_OUT = "/kaggle/working/hybrid_output.mp4"

conf = 0.25
iou  = 0.45
img_size = 640

# -------------------
# Helper: run hybrid on a BGR frame
# -------------------
def detect_bgr_with_detector(detector, frame_bgr, conf=0.25, iou=0.45, img_size=640):
    from utils.general import non_max_suppression, scale_coords
    from utils.datasets import letterbox

    device = detector.device  # from your HybridDetector2

    img0 = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    h0, w0 = img0.shape[:2]

    img = letterbox(img0, img_size, stride=32, auto=False)[0]
    img = img.transpose((2,0,1))
    img = np.ascontiguousarray(img)

    img_t = torch.from_numpy(img).to(device).float() / 255.0
    if img_t.ndimension() == 3:
        img_t = img_t.unsqueeze(0)

    with torch.no_grad():
        pred = detector.yolo(img_t, augment=False)[0]
        pred = non_max_suppression(pred, conf_thres=conf, iou_thres=iou)[0]

    results = []
    if pred is None or len(pred) == 0:
        return results

    pred[:, :4] = scale_coords(img_t.shape[2:], pred[:, :4], (h0, w0))

    pil0 = Image.fromarray(img0)
    with torch.no_grad():
        for det in pred:
            x1,y1,x2,y2,conf_det,cls_det = det.tolist()
            x1,y1,x2,y2 = map(int, [x1,y1,x2,y2])

            x1 = max(0, min(x1, w0-1))
            y1 = max(0, min(y1, h0-1))
            x2 = max(0, min(x2, w0-1))
            y2 = max(0, min(y2, h0-1))
            if x2 <= x1 or y2 <= y1:
                continue

            crop = pil0.crop((x1,y1,x2,y2))
            x = detector.tf(crop).unsqueeze(0).to(device)

            logits = detector.resnet(x)
            prob = torch.softmax(logits, dim=1)[0]
            cls2 = int(prob.argmax().item())
            conf2 = float(prob[cls2].item())

            cls_det = int(cls_det)
            results.append({
                "bbox": [x1,y1,x2,y2],
                "yolo_cls": cls_det,
                "yolo_name": detector.names[cls_det],
                "yolo_conf": float(conf_det),
                "resnet_cls": cls2,
                "resnet_name": detector.cls_names[cls2],
                "resnet_conf": conf2
            })

    return results

# -------------------
# Process video
# -------------------
cap = cv2.VideoCapture(VIDEO_IN)
assert cap.isOpened(), f"Could not open video: {VIDEO_IN}"

fps = cap.get(cv2.CAP_PROP_FPS) or 25
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (W, H))

frame_i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break

    results = detect_bgr_with_detector(detector, frame, conf=conf, iou=iou, img_size=img_size)

    # draw results
    for r in results:
        x1,y1,x2,y2 = r["bbox"]
        label = f'{r["yolo_name"]} {r["yolo_conf"]:.2f} | {r["resnet_name"]} {r["resnet_conf"]:.2f}'
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(frame, label, (x1, max(20, y1-7)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

    writer.write(frame)

    frame_i += 1
    if frame_i % 200 == 0:
        print("processed frames:", frame_i)

cap.release()
writer.release()

print("✅ Saved:", VIDEO_OUT)


processed frames: 200
processed frames: 400
processed frames: 600
processed frames: 800
processed frames: 1000
processed frames: 1200
processed frames: 1400
processed frames: 1600
processed frames: 1800
processed frames: 2000
processed frames: 2200
processed frames: 2400
processed frames: 2600
processed frames: 2800


KeyboardInterrupt: 

In [67]:
VIDEO_IN = "/kaggle/input/videodetectbyhybrid/1.mp4"
CLIP_OUT = "/kaggle/working/clip_30s.mp4"

!ffmpeg -y -ss 00:00:00 -t 00:00:30 -i "{VIDEO_IN}" -c copy "{CLIP_OUT}"
print("✅ Saved:", CLIP_OUT)


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [68]:
import os
import cv2
import numpy as np
import torch
from PIL import Image

VIDEO_IN  = "/kaggle/working/clip_30s.mp4"   # <-- change this
VIDEO_OUT = "/kaggle/working/hybrid_output_8fps.mp4"

conf = 0.25
iou  = 0.45
img_size = 640
TARGET_FPS = 8  # change to 5, 8, 10 etc.

def detect_bgr_with_detector(detector, frame_bgr, conf=0.25, iou=0.45, img_size=640):
    from utils.general import non_max_suppression, scale_coords
    from utils.datasets import letterbox

    device = detector.device

    img0 = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    h0, w0 = img0.shape[:2]

    img = letterbox(img0, img_size, stride=32, auto=False)[0]
    img = img.transpose((2,0,1))
    img = np.ascontiguousarray(img)

    img_t = torch.from_numpy(img).to(device).float() / 255.0
    if img_t.ndimension() == 3:
        img_t = img_t.unsqueeze(0)

    with torch.no_grad():
        pred = detector.yolo(img_t, augment=False)[0]
        pred = non_max_suppression(pred, conf_thres=conf, iou_thres=iou)[0]

    results = []
    if pred is None or len(pred) == 0:
        return results

    pred[:, :4] = scale_coords(img_t.shape[2:], pred[:, :4], (h0, w0))

    pil0 = Image.fromarray(img0)
    with torch.no_grad():
        for det in pred:
            x1,y1,x2,y2,conf_det,cls_det = det.tolist()
            x1,y1,x2,y2 = map(int, [x1,y1,x2,y2])

            x1 = max(0, min(x1, w0-1))
            y1 = max(0, min(y1, h0-1))
            x2 = max(0, min(x2, w0-1))
            y2 = max(0, min(y2, h0-1))
            if x2 <= x1 or y2 <= y1:
                continue

            crop = pil0.crop((x1,y1,x2,y2))
            x = detector.tf(crop).unsqueeze(0).to(device)

            logits = detector.resnet(x)
            prob = torch.softmax(logits, dim=1)[0]
            cls2 = int(prob.argmax().item())
            conf2 = float(prob[cls2].item())

            cls_det = int(cls_det)
            results.append({
                "bbox": [x1,y1,x2,y2],
                "yolo_name": detector.names[cls_det],
                "yolo_conf": float(conf_det),
                "resnet_name": detector.cls_names[cls2],
                "resnet_conf": conf2
            })
    return results

cap = cv2.VideoCapture(VIDEO_IN)
assert cap.isOpened(), f"Could not open video: {VIDEO_IN}"

src_fps = cap.get(cv2.CAP_PROP_FPS) or 25
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

stride = max(1, int(round(src_fps / TARGET_FPS)))
out_fps = src_fps / stride

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_OUT, fourcc, out_fps, (W, H))

frame_i = 0
processed = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    if frame_i % stride == 0:
        results = detect_bgr_with_detector(detector, frame, conf=conf, iou=iou, img_size=img_size)
        processed += 1

        for r in results:
            x1,y1,x2,y2 = r["bbox"]
            label = f'{r["yolo_name"]} {r["yolo_conf"]:.2f} | {r["resnet_name"]} {r["resnet_conf"]:.2f}'
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
            cv2.putText(frame, label, (x1, max(20, y1-7)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

        writer.write(frame)

    frame_i += 1
    if frame_i % 200 == 0:
        print(f"Read frames: {frame_i} | Processed frames: {processed} | stride={stride} | out_fps={out_fps:.2f}")

cap.release()
writer.release()

print("✅ Saved:", VIDEO_OUT)


Read frames: 200 | Processed frames: 50 | stride=4 | out_fps=7.50
Read frames: 400 | Processed frames: 100 | stride=4 | out_fps=7.50
Read frames: 600 | Processed frames: 150 | stride=4 | out_fps=7.50
Read frames: 800 | Processed frames: 200 | stride=4 | out_fps=7.50
✅ Saved: /kaggle/working/hybrid_output_8fps.mp4
